In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

import os
import json

sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("All libraries imported successfully.")



All libraries imported successfully.


In [5]:
df = pd.read_csv('/Users/tahminasahar/earthquake-risk-afghanistan/data/usgs_afghanistan_features.csv')

print(f"Loaded {len(df)} earthquake records from the dataset.")
print(f"Total columns: {len(df.columns)}")
print(f"\nVerfying key features exist: ")

requried = ['mag', 'depth', 'depth_risk_score', 'mag_risk_score',
            'energy_proxy', 'dist_to_kabul', 'combined_risk_score']
for col in requried:
    if col in df.columns:
        print(f"✅ {col} exists in the dataset.")
    else:
        print(f"❌ {col} is missing from the dataset.")

Loaded 8250 earthquake records from the dataset.
Total columns: 39

Verfying key features exist: 
✅ mag exists in the dataset.
✅ depth exists in the dataset.
✅ depth_risk_score exists in the dataset.
✅ mag_risk_score exists in the dataset.
✅ energy_proxy exists in the dataset.
✅ dist_to_kabul exists in the dataset.
✅ combined_risk_score exists in the dataset.


In [6]:
BASE = '/Users/tahminasahar/earthquake-risk-afghanistan'
df = pd.read_csv(f'{BASE}/data/usgs_afghanistan_features.csv')

def assign_risk_label(score):
    if score >= 0.35:
        return 'HIGH'
    elif score >= 0.15:
        return 'Medium'
    else:
        return 'Low'

df['risk_label'] = df['combined_risk_score'].apply(assign_risk_label)

feature_columns = ['mag', 'depth', 'depth_risk_score', 'energy_proxy',
                   'dist_to_kabul', 'lat_grid', 'lon_grid']

x = df[feature_columns]
y = df['risk_label']

X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)

rf_model.fit(X_train, Y_train)

y_pred = rf_model.predict(X_test)

print(f"Accuracy: {accuracy_score(Y_test, y_pred):.1%}")
print(classification_report(Y_test, y_pred))

joblib.dump(rf_model, f'{BASE}/models/random_forest_risk_classifier.pkl')
with open(f'{BASE}/models/feature_columns.json', 'w') as f:
          json.dump(feature_columns, f)
print("✅ Model + feature list saved")


Accuracy: 100.0%
              precision    recall  f1-score   support

        HIGH       1.00      1.00      1.00       104
         Low       1.00      1.00      1.00       421
      Medium       1.00      1.00      1.00      1125

    accuracy                           1.00      1650
   macro avg       1.00      1.00      1.00      1650
weighted avg       1.00      1.00      1.00      1650

✅ Model + feature list saved
